# 3.3 Core Components — Models, Prompts & Output Parsers

## Playground Notebook

Every LLM application is built from three foundational building blocks:

| Component | Role | Analogy |
|-----------|------|---------|
| **Models** | The AI brain | The stove (heat source) |
| **Prompts** | The instructions | The recipe |
| **Output Parsers** | The response formatter | The plating |

Together they form the **minimum viable chain** — the smallest meaningful unit in LangChain:

```
Prompt Template  →  Chat Model  →  Output Parser
```

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [ ]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. Component 1: Models — The AI Brain

LangChain provides a **unified interface** over all LLM providers. You write application logic once and swap models by changing a single line.

### Two Model Types

| Type | Interface | Status |
|------|-----------|--------|
| **LLMs** (Legacy) | Text in → Text out | Largely deprecated |
| **Chat Models** (Modern) | Messages in → Message out | The standard |

### Message Types

| Message Type | Role | Purpose |
|-------------|------|----------|
| `SystemMessage` | System | Sets behavior, persona, constraints |
| `HumanMessage` | User | The user's input or question |
| `AIMessage` | Assistant | The model's prior response |
| `ToolMessage` | Tool | Results from tool/function calls |

### The Unified Model Interface

Every Chat Model supports these methods:

| Method | Purpose |
|--------|---------|
| `invoke()` | Send messages, get complete response |
| `stream()` | Token-by-token streaming |
| `batch()` | Process multiple inputs in parallel |
| `bind_tools()` | Attach tool definitions for function calling |
| `with_structured_output()` | Force output into a schema |

### Experiment 1A: Message Types in Action

In [2]:
# Structured messages — typed, self-documenting, role-aware
messages = [
    SystemMessage(content="You are a concise technical explainer. Max 2 sentences."),
    HumanMessage(content="What is a Chat Model in LangChain?")
]

response = llm.invoke(messages)

print(f"Response type : {type(response).__name__}")
print(f"Response role : {response.type}")
print(f"Content       : {response.content}")

Response type : AIMessage
Response role : ai
Content       : In LangChain, a Chat Model is a type of transformer-based language model that uses the Sentence-BERT (sBERT) architecture to generate human-like responses to user input. It's designed to be a conversational AI model, trained on a large dataset of conversations or dialogues.

The Chat Model in LangChain is typically fine-tuned from pre-trained sBERT models and can be used for various NLP tasks such as:

* Response generation
* Conversation management
* Text classification

LangChain's Chat Model provides several benefits, including:

* **Efficient computation**: The model is optimized for fast inference and can handle large volumes of conversations.
* **Improved accuracy**: The fine-tuned model has achieved state-of-the-art performance on various NLP tasks.

The Chat Model in LangChain can be integrated with other LangChain components to build more complex conversational AI systems, such as:

* Dialogue management
* Entity rec

### Experiment 1B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [3]:
# --- invoke(): single input, full response ---
print("=" * 50)
print("invoke() — Full response at once")
print("=" * 50)

result = llm.invoke([
    SystemMessage(content="Answer in one sentence."),
    HumanMessage(content="What is the Runnable protocol?")
])
print(result.content)

invoke() — Full response at once
The Runnable interface is a part of the Java API (Java Application Programming Interface) that allows you to represent an executable unit of code, such as a class or method, that can be executed independently.

In other words, a Runnable is an object that contains a reference to another object or a block of code that can be executed by the JVM (Java Virtual Machine). This interface is typically used when you want to pass a task or a piece of code to another thread or process, without having to create a new instance of a class or method.

A Runnable has two main methods:

1. `void run()` - This method contains the actual code that will be executed by the JVM.
2. `Object getDeclaringClass()` - This method returns the class that declares the Runnable object.

The main use cases for Runnables are:

* **Executor framework**: In Java, a Executor is an object that manages a pool of threads and allows you to submit tasks (i.e., Runnables) to be executed by thos

In [4]:
# --- stream(): token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in llm.stream("Name 3 LangChain message types. One line each."):
    print(chunk.content, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are three LangChain message types:

1. **TextMessage**: a message that contains plain text.
2. **EmbeddingMessage**: a message that contains numerical embeddings (e.g., from a sentence transformer model).
3. **EntityMessage**: a message that contains named entities (e.g., a person's name or organization).


In [5]:
# --- batch(): multiple inputs at once ---
print("=" * 50)
print("batch() — Process multiple inputs")
print("=" * 50)

questions = [
    "What is SystemMessage? One sentence.",
    "What is HumanMessage? One sentence.",
    "What is AIMessage? One sentence."
]

results = llm.batch(questions)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

batch() — Process multiple inputs
Q: What is SystemMessage? One sentence.
A: SystemMessage is a module within PyTorch that provides a simple way to send messages from one part of your code to another, allowing for easier debugging and communication between different parts of your program.

Q: What is HumanMessage? One sentence.
A: HumanMessage is an open-source, pre-trained sentence transformer model specifically designed for human-centered applications such as sentiment analysis, opinion mining, and human-like text generation.

Q: What is AIMessage? One sentence.
A: AIMessage is a pre-trained sentence transformer model specifically designed to generate informative, coherent, and engaging messages based on input prompts, similar to chatbots or virtual assistants.



### Experiment 1C: Model Configuration Parameters

| Parameter | Default | Purpose |
|-----------|---------|----------|
| `temperature` | 0.7 | Randomness: 0 = deterministic, 1 = creative |
| `max_tokens` | varies | Maximum tokens in the response |
| `top_p` | 1.0 | Nucleus sampling — alternative to temperature |
| `stop` | None | Stop sequences that halt generation |

**Temperature Guidance:**
- **0.0** — Factual tasks (classification, extraction, code)
- **0.3–0.5** — Balanced (summarization, Q&A)
- **0.7–0.9** — Creative (brainstorming, writing)
- **1.0+** — Maximum creativity (poetry, experimental)

In [6]:
# Temperature comparison — same prompt, different randomness
prompt = "Give me a one-sentence tagline for a coffee shop."

configs = [
    {"label": "Deterministic (temp=0.0)", "temp": 0.0},
    {"label": "Balanced (temp=0.5)",      "temp": 0.5},
    {"label": "Creative (temp=1.0)",       "temp": 1.0},
]

for cfg in configs:
    model = ChatOllama(model=MODEL, temperature=cfg["temp"])
    # Run twice to show consistency vs variety
    r1 = model.invoke(prompt).content
    r2 = model.invoke(prompt).content
    print(f"\n{cfg['label']}")
    print(f"  Run 1: {r1}")
    print(f"  Run 2: {r2}")
    print(f"  Same? {'Yes' if r1 == r2 else 'No'}")


Deterministic (temp=0.0)
  Run 1: "Wake up to the perfect blend of flavor, comfort, and community."
  Run 2: "Wake up to the perfect blend of flavor, comfort, and community."
  Same? Yes

Balanced (temp=0.5)
  Run 1: "Wake up to the art of living, one cup at a time."
  Run 2: "Brewing connections, one cup at a time."
  Same? No

Creative (temp=1.0)
  Run 1: "Perk up your day, one cup at a time."
  Run 2: "Wake up to the art of conversation"
  Same? No


### Experiment 1D: Provider Swappability — The Power of Abstraction

LangChain's greatest strength: **swap providers by changing one line**.

```python
# All these use the SAME interface
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatOpenAI(model="gpt-4o-mini")              # OpenAI
llm = ChatAnthropic(model="claude-3-5-sonnet")      # Anthropic
llm = ChatOllama(model="llama3.2:3b")               # Local
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")  # Google

# Your chain logic stays IDENTICAL
chain = prompt | llm | parser
```

This means you can:
- A/B test different models with zero code changes
- Switch to cheaper models for non-critical tasks
- Fall back to alternatives during outages
- Migrate between providers as pricing changes

In [7]:
# Demonstrate swappability with different Ollama configurations
# (Same pattern applies across OpenAI, Anthropic, Google, etc.)

parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. One sentence max."),
    ("human", "What is LangChain?")
])

# Two different "providers" (same model, different configs)
precise_llm = ChatOllama(model=MODEL, temperature=0.0)
creative_llm = ChatOllama(model=MODEL, temperature=1.0)

# SAME chain structure, DIFFERENT models plugged in
chain_a = prompt | precise_llm | parser
chain_b = prompt | creative_llm | parser

print("Precise chain:")
print(chain_a.invoke({}))

print("\nCreative chain:")
print(chain_b.invoke({}))

print("\nChain logic is identical — only the model changed!")

Precise chain:
LangChain is an open-source Python library that provides a simple and flexible way to build and integrate various natural language processing (NLP) pipelines, including those that utilize sentence transformers.

LangChain was designed to make it easier for developers to work with NLP models and libraries, such as Hugging Face's Transformers and SentenceTransformer, by providing a high-level interface for building and chaining together different components of an NLP pipeline.

Some key features of LangChain include:

* **Pipeline building**: LangChain allows you to build complex NLP pipelines by chaining together individual components, such as text preprocessing, tokenization, and model inference.
* **Model integration**: LangChain provides a simple way to integrate popular NLP models, including sentence transformers, into your pipeline.
* **Data loading and caching**: LangChain includes tools for loading and caching data, which can help improve the performance of your pi

---

## 2. Component 2: Prompts — The Instructions

Prompts are the instructions you send to the model. LangChain's templating system makes them **reusable, parameterized, composable, and type-safe**.

### Why Not Just Use Raw Strings?

| Problem with Raw Strings | LangChain Solution |
|--------------------------|--------------------|
| Hard to reuse | Templates are objects — import and share |
| Hard to test | Templates are pure functions — test without LLM |
| Hard to maintain | Change template once, used everywhere |
| No type safety | Input validation catches missing variables |
| No composability | Templates combine with `+` and `\|` |

### Prompt Template Types

| Template | Use Case | Frequency |
|----------|----------|----------|
| `PromptTemplate` | Simple string with variables | Occasional |
| `ChatPromptTemplate` | Role-based messages with variables | **90%+ of the time** |
| `MessagesPlaceholder` | Dynamic message injection (history) | Common |
| `FewShotChatMessagePromptTemplate` | Example-based learning | When needed |

### Experiment 2A: PromptTemplate — Simple String Templates

In [8]:
# PromptTemplate — basic string with {variable} placeholders
simple_prompt = PromptTemplate.from_template(
    "Explain {topic} to a {audience} in {length}."
)

# See the template info
print(f"Template: {simple_prompt.template}")
print(f"Variables: {simple_prompt.input_variables}")

# Format it
formatted = simple_prompt.invoke({
    "topic": "APIs",
    "audience": "beginner",
    "length": "2 sentences"
})

print(f"\nFormatted: {formatted.text}")

# Send to model
response = llm.invoke(formatted.text)
print(f"\nResponse: {response.content}")

Template: Explain {topic} to a {audience} in {length}.
Variables: ['audience', 'length', 'topic']

Formatted: Explain APIs to a beginner in 2 sentences.

Response: An API, or Application Programming Interface, is like a messenger that helps different computer systems communicate with each other, allowing them to share information and exchange data in a standardized way. Think of an API as a set of instructions that says "if you want to get this piece of information from the database, follow these steps" - it acts as a bridge between two systems, enabling them to work together seamlessly.


### Experiment 2B: ChatPromptTemplate — The Modern Standard

Produces a list of **structured messages with roles**. This is what you'll use 90%+ of the time.

In [9]:
# ChatPromptTemplate with role-based messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in {style} style."),
    ("human", "{question}")
])

# See template info
print(f"Variables: {chat_prompt.input_variables}")

# Format with values
messages = chat_prompt.invoke({
    "role": "Python tutor",
    "style": "concise",
    "question": "What is a list comprehension?"
})

print("\nFormatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

# Send to model
response = llm.invoke(messages)
print(f"\nResponse: {response.content}")

Variables: ['question', 'role', 'style']

Formatted messages:
  [SystemMessage] You are a Python tutor. Always respond in concise style.
  [HumanMessage] What is a list comprehension?

Response: A list comprehension is a concise way to create lists in Python by performing an operation on each element of an iterable (such as a list, tuple, or set).

The general syntax for a list comprehension is:
```python
new_list = [expression for variable in iterable]
```
Here's a breakdown:

* `variable` is the temporary name given to each element in the `iterable`.
* `expression` is the operation performed on each `variable`. This can be any valid Python expression.
* `new_list` is the resulting list created by applying the `expression` to each element in the `iterable`.

Examples:

```python
# Double each number in a list
numbers = [1, 2, 3, 4, 5]
doubled_numbers = [x * 2 for x in numbers]
print(doubled_numbers)  # [2, 4, 6, 8, 10]

# Filter out even numbers from a list
numbers = [1, 2, 3, 4, 5]
o

In [10]:
# Reuse the SAME template with different inputs
scenarios = [
    {"role": "pirate captain", "style": "pirate", "question": "What is recursion?"},
    {"role": "formal professor", "style": "academic", "question": "What is recursion?"},
]

for s in scenarios:
    print(f"\n{'=' * 50}")
    print(f"Role: {s['role']}")
    print(f"{'=' * 50}")
    result = llm.invoke(chat_prompt.invoke(s))
    display(Markdown(result.content))


Role: pirate captain


Arrr, recursion be a powerful tool fer computin' complex problems, but it can also be a mighty curse if ye don't use it wisely.

Recursion be a process where a function calls itself repeatedly until it reaches a base case that stops the recursion. It's like a treasure hunt where ye follow a trail o' clues to find the loot, and each clue leads ye back to the startin' point, but with more information to help ye solve the problem.

Here be some key characteristics o' recursion:

1. **Base case**: A condition that stops the recursion when met.
2. **Recursive call**: The function calls itself with new arguments or a modified version o' itself.
3. **Termination condition**: A condition that marks the end o' the recursion, where the function returns to its caller.

Here be an example o' a recursive function in Python:
```python
def factorial(n):
    if n == 0:  # base case
        return 1
    else:
        return n * factorial(n-1)  # recursive call
```
In this example, the `factorial` function calls itself with decreasing values o' `n` until it reaches the base case (`n == 0`). The results o' each recursive call are then multiplied together to give the final result.

Recursion has some advantages:

* **Easy to implement**: Recursive functions can be easier to understand and write than iterative solutions.
* **Faster in some cases**: Recursion can be faster than iteration when dealin' with problems that have a small number o' recursive calls.

However, recursion also has some disadvantages:

* **Memory usage**: Each recursive call creates a new stack frame, which can consume a lot o' memory if the function is called too many times.
* **Risk o' stack overflow**: If the function is called too deeply, it can cause a stack overflow error.

So hoist the sails and set course fer recursion, but be sure to navigate through the waters o' memory management and termination conditions wisely!


Role: formal professor


Recursion is a fundamental concept in computer science and mathematics that involves the repeated application of the same algorithm or function to solve a problem.

**Definition:**

Recursion is a technique where a function calls itself as a subroutine, which can be used to solve a problem by breaking it down into smaller instances of the same problem. The recursive function makes multiple recursive calls until the solution to the original problem is found.

**Key characteristics:**

1. **Base case:** A recursive function must have a base case that stops the recursion when the problem size becomes too small to be solved.
2. **Recursive call:** The recursive function calls itself with smaller inputs, solving the same problem.
3. **Memoization or caching:** To avoid redundant computation, some recursive functions may use memoization or caching to store and reuse previously computed results.

**Types of recursion:**

1. **Direct recursion:** A function calls itself directly without any intermediate steps.
2. **Indirect recursion:** A function calls another function that in turn calls the original function.
3. **Tail recursion:** The recursive call is made as part of a larger expression, rather than at the end of a statement.

**Examples:**

1. **Factorial calculation:** `fact(n)` = `n * fact(n-1)`
2. **Tree traversal:** Traverse a binary tree by visiting each node and recursively traversing its children.
3. **Fibonacci sequence:** Calculate Fibonacci numbers using recursion, where `fib(n)` = `n + fib(n-1)`

**Advantages:**

1. **Elegant solutions:** Recursion can provide an elegant and concise way to solve problems that have a recursive structure.
2. **Efficient memory usage:** By reusing previously computed results, recursion can be more memory-efficient than iterative solutions.

**Disadvantages:**

1. **Increased complexity:** Recursive functions can be harder to understand and debug due to the nested calls.
2. **Performance issues:** Deep recursion can lead to stack overflows or performance degradation due to repeated function calls.

**Recursion in programming languages:**

Most programming languages support recursive functions, including Python, Java, C++, and many others. Recursion is often used in algorithms for problems like dynamic programming, graph traversal, and tree operations.

In conclusion, recursion is a powerful technique for solving problems by breaking them down into smaller instances of the same problem. While it has its advantages, careful consideration must be given to avoid performance issues or complexity that may make the code harder to understand.

### Experiment 2C: MessagesPlaceholder — Dynamic Conversation History

`MessagesPlaceholder` creates a **slot** where a variable-length list of messages can be injected at runtime. Essential for conversation memory.

In [11]:
# Template with a slot for conversation history
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

print(f"Variables: {history_prompt.input_variables}")

# Simulate conversation history
fake_history = [
    HumanMessage(content="What is 2 + 2?"),
    AIMessage(content="2 + 2 = 4."),
    HumanMessage(content="Now multiply that by 3."),
    AIMessage(content="4 x 3 = 12."),
]

# Inject history + new question
messages = history_prompt.invoke({
    "chat_history": fake_history,
    "question": "What was my first question?"
})

print("\nFormatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

# The model can now "remember" the conversation
response = llm.invoke(messages)
print(f"\nResponse: {response.content}")

Variables: ['chat_history', 'question']

Formatted messages:
  [SystemMessage] You are a helpful math tutor. Be concise.
  [HumanMessage] What is 2 + 2?
  [AIMessage] 2 + 2 = 4.
  [HumanMessage] Now multiply that by 3.
  [AIMessage] 4 x 3 = 12.
  [HumanMessage] What was my first question?

Response: Your first question was about "Sentence Transformer".


### Experiment 2D: Working Conversation with MessagesPlaceholder

In [12]:
# Full working conversation using MessagesPlaceholder
convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Keep answers to 1-2 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = convo_prompt | llm | StrOutputParser()

# Build history turn by turn
history = []
turns = [
    "My name is Alex.",
    "I'm learning LangChain.",
    "What is my name and what am I learning?"
]

for i, user_input in enumerate(turns, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}")
    print(f"{'=' * 50}")
    print(f"User: {user_input}")

    response = chain.invoke({"history": history, "input": user_input})
    print(f"AI:   {response}")

    # Save to history
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))

print(f"\nHistory length: {len(history)} messages")
print("MessagesPlaceholder injected the full history each turn!")


Turn 1
User: My name is Alex.
AI:   Nice to meet you, Alex! Is there something I can help you with or would you like to chat?

Turn 2
User: I'm learning LangChain.
AI:   LangChain is an open-source Python library for building and deploying conversational AI models. It provides a flexible framework for integrating various NLP models, including transformer-based architectures. How's your progress with LangChain so far, Alex?

Turn 3
User: What is my name and what am I learning?
AI:   Your name is Alex, and you are currently learning about LangChain.

History length: 6 messages
MessagesPlaceholder injected the full history each turn!


# StructuredOutputParser was removed in LangChain v1.x
# Use JsonOutputParser with field-based prompting instead (same result)

json_field_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You provide technical facts. Respond ONLY with valid JSON. No markdown, no extra text.\n"
     "Use exactly these keys: \"language\", \"year_created\", \"best_for\" (one sentence)."),
    ("human", "Tell me about {language}.")
])

json_field_chain = json_field_prompt | llm | JsonOutputParser()

result = json_field_chain.invoke({"language": "Python"})

print(f"Type:    {type(result).__name__}")
for key, value in result.items():
    print(f"  {key}: {value}")

In [13]:
# Define example input-output pairs
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

# Template for formatting each example as a human-AI exchange
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Build the few-shot template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Wrap in a full ChatPromptTemplate
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You give the opposite of the word provided. One word only."),
    few_shot_prompt,
    ("human", "{input}")
])

chain = final_prompt | llm | StrOutputParser()

# Test with new inputs
test_words = ["bright", "heavy", "loud", "hot"]

for word in test_words:
    result = chain.invoke({"input": word})
    print(f"  {word} → {result}")

  bright → dark
  heavy → light
  loud → quiet
  hot → cold


### Experiment 2F: Partial Prompts — Pre-Fill Variables

Pre-fill some variables at setup time, fill the rest at runtime.

In [14]:
from datetime import datetime

# Template with a mix of static and runtime variables
base_prompt = PromptTemplate.from_template(
    "Today is {date}. You are {assistant_name}. Answer: {question}"
)

# Partial — pre-fill the static variables now
partial_prompt = base_prompt.partial(
    date=datetime.now().strftime("%Y-%m-%d"),
    assistant_name="CodeBot"
)

print(f"Original variables: {base_prompt.input_variables}")
print(f"After partial:      {partial_prompt.input_variables}")

# At runtime — only provide the remaining variable
formatted = partial_prompt.invoke({"question": "What day is it?"})
print(f"\nFormatted: {formatted.text}")

response = llm.invoke(formatted.text)
print(f"Response:  {response.content}")

Original variables: ['assistant_name', 'date', 'question']
After partial:      ['question']

Formatted: Today is 2026-03-01. You are CodeBot. Answer: What day is it?
Response:  It's March 1st, 2026.


### Prompt Template Best Practices

| Practice | Description |
|----------|-------------|
| Always use templates over raw strings | Enables testing, reuse, and versioning |
| Name variables descriptively | `{user_question}` > `{q}` |
| Keep system prompts separate | Easy to swap or A/B test |
| Use partial variables for static context | Pre-fill date, model name, app version |
| Test templates independently | Templates are pure functions — test without the LLM |
| Use few-shot for complex tasks | When zero-shot struggles, add 2–3 examples |

---

## 3. Component 3: Output Parsers — The Response Formatter

LLMs return **raw text**. Output parsers transform that text into **structured, typed data** your application can use.

### The Core Problem

When you ask an LLM to "return JSON", you might get:
- Perfect JSON: `{"name": "John", "age": 30}`
- JSON wrapped in markdown code fences
- JSON with preamble text before it
- Slightly malformed JSON
- A natural language response instead

### The Output Parser Interface

Every parser implements two key methods:

| Method | Purpose |
|--------|---------|
| `get_format_instructions()` | Returns text to inject into the prompt telling the model how to format output |
| `parse()` | Takes the raw LLM string and converts it into structured data |

### Parser Comparison

| Parser | Output Type | Best For |
|--------|-------------|----------|
| `StrOutputParser` | `str` | Plain text responses |
| `CommaSeparatedListOutputParser` | `list` | Simple lists |
| `JsonOutputParser` | `dict` | Quick structured data |
| `PydanticOutputParser` | Pydantic model | Production apps with validation |
| `StructuredOutputParser` | `dict` | Prototyping structured output |
| `EnumOutputParser` | enum value | Classification tasks |

### Experiment 3A: StrOutputParser — The Basics

In [15]:
# Without parser — you get an AIMessage object
prompt = ChatPromptTemplate.from_messages([
    ("system", "Be concise."),
    ("human", "What is an output parser?")
])

raw_chain = prompt | llm
raw_result = raw_chain.invoke({})

print(f"Without parser:")
print(f"  Type:    {type(raw_result).__name__}")
print(f"  Content: {raw_result.content}")

print()

# With StrOutputParser — you get a clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({})

print(f"With StrOutputParser:")
print(f"  Type:    {type(parsed_result).__name__}")
print(f"  Content: {parsed_result}")

Without parser:
  Type:    AIMessage
  Content: In natural language processing (NLP), a **parser** is a component that analyzes a sequence of tokens (such as words or characters) to identify its grammatical structure, meaning, or context.

An **output parser** is a type of parser that generates an explicit output representation of the input text, such as:

1. **Named entity recognition**: Identifying and labeling named entities in the text, such as people, organizations, locations, etc.
2. **Part-of-speech tagging**: Identifying the part of speech (e.g., noun, verb, adjective, adverb) for each word or token in the input text.
3. **Dependency parsing**: Analyzing the grammatical structure of the sentence and representing it as a graph of relationships between tokens.
4. **Semantic role labeling**: Identifying the roles played by entities in a clause or sentence (e.g., "agent", "patient", "theme").

Output parsers are used to generate structured representations of text data, which can be

### Experiment 3B: CommaSeparatedListOutputParser — Get a Python List

In [16]:
list_parser = CommaSeparatedListOutputParser()

# See what format instructions look like
print("Format instructions:")
print(f"  {list_parser.get_format_instructions()}")

# Build chain with format instructions injected into the prompt
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items separated by commas. No numbering, no extra text. {format_instructions}"),
    ("human", "List 5 {category}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({
    "category": "popular programming languages",
    "format_instructions": list_parser.get_format_instructions()
})

print(f"\nType:   {type(result).__name__}")
print(f"Result: {result}")
print(f"First:  {result[0]}")
print(f"Count:  {len(result)}")

Format instructions:
  Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`

Type:   list
Result: ['Python', 'JavaScript', 'Java', 'C++', 'TypeScript']
First:  Python
Count:  5


### Experiment 3C: JsonOutputParser — Structured Data Extraction

In [17]:
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Always respond with valid JSON only. "
     "No markdown, no explanation — just the JSON object."),
    ("human",
     'Provide info about {topic}. '
     'Return JSON with keys: "name", "type", "description" (1 sentence).')
])

json_chain = json_prompt | llm | json_parser

result = json_chain.invoke({"topic": "Python programming language"})

print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")

if isinstance(result, dict):
    print("\nParsed fields:")
    for key, value in result.items():
        print(f"  {key}: {value}")

Type:   dict
Result: {'name': 'Python', 'type': 'high-level', 'description': 'A high-level, interpreted programming language with a focus on simplicity and ease of use.'}

Parsed fields:
  name: Python
  type: high-level
  description: A high-level, interpreted programming language with a focus on simplicity and ease of use.


### Experiment 3D: PydanticOutputParser — Production-Grade Schema Validation

The **gold standard** for production applications. Full type validation, default values, and clear error messages.

In [18]:
from pydantic import BaseModel, Field

# Define the expected output schema
class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    genre: str = Field(description="The movie genre")
    rating: int = Field(description="Rating from 1-10")
    summary: str = Field(description="One sentence summary")

# Create parser from the schema
pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

# See the auto-generated format instructions
print("Format instructions (auto-generated from Pydantic model):")
print(pydantic_parser.get_format_instructions())

Format instructions (auto-generated from Pydantic model):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The movie title", "title": "Title", "type": "string"}, "genre": {"description": "The movie genre", "title": "Genre", "type": "string"}, "rating": {"description": "Rating from 1-10", "title": "Rating", "type": "integer"}, "summary": {"description": "One sentence summary", "title": "Summary", "type": "string"}}, "required": ["title", "genre", "rating", "summary"]}
```


In [19]:
# Build chain with format instructions
pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a movie reviewer. Respond ONLY with valid JSON matching the schema. "
     "No markdown, no extra text.\n{format_instructions}"),
    ("human", "Review the movie: {movie}")
])

pydantic_chain = pydantic_prompt | llm | pydantic_parser

result = pydantic_chain.invoke({
    "movie": "The Matrix",
    "format_instructions": pydantic_parser.get_format_instructions()
})

print(f"Type:    {type(result).__name__}")
print(f"Title:   {result.title}")
print(f"Genre:   {result.genre}")
print(f"Rating:  {result.rating}")
print(f"Summary: {result.summary}")

Type:    MovieReview
Title:   The Matrix
Genre:   Action, Sci-Fi, Thriller
Rating:  9
Summary: In a dystopian future, humanity is unknowingly trapped within a simulated reality called the Matrix. A computer hacker named Neo discovers the truth and must join a group of rebels to fight against the machines and free humanity.


In [20]:
# Compare: Multiple movies through the same chain
movies = ["Inception", "The Godfather", "Toy Story"]

for movie in movies:
    print(f"\n{'=' * 50}")
    try:
        review = pydantic_chain.invoke({
            "movie": movie,
            "format_instructions": pydantic_parser.get_format_instructions()
        })
        print(f"  {review.title} | {review.genre} | {review.rating}/10")
        print(f"  {review.summary}")
    except Exception as e:
        print(f"  Parse error for '{movie}': {e}")


  Inception | Science Fiction, Action, Thriller | 8/10
  A thief who specializes in entering people\u2019s dreams and stealing their secrets is given a chance to redeem himself by performing an even greater task: planting an idea in someone\u2019s mind.

  The Godfather | Crime, Drama | 9/10
  In the world of organized crime, the Corleone family rules with an iron fist. When Don Vito Corleone is nearly assassinated, his youngest son Michael takes up the mantle and becomes a powerful mafia leader.

  Toy Story | Animation, Adventure, Comedy | 8/10
  A toy cowboy named Woody is jealous when a new toy, Buzz Lightyear, becomes his owner’s favorite. However, when they are accidentally left behind by their owner and become lost, the two must put aside their differences and work together to find their way home.


### Experiment 3E: StructuredOutputParser — Quick Prototyping

In [27]:
from langchain_core.output_parsers import StrOutputParser
import json

# Define the expected fields (replaces ResponseSchema)
fields = {
    "language": "The programming language name",
    "year_created": "Year the language was created",
    "best_for": "What the language is best used for, in one sentence",
}

# Build format instructions string (replaces get_format_instructions())
format_instructions = (
    "Respond with valid JSON only. No markdown, no extra text.\n"
    "Use exactly these keys:\n"
    + "\n".join(f'  "{k}": {v}' for k, v in fields.items())
)

print("Format instructions:")
print(format_instructions)


Format instructions:
Respond with valid JSON only. No markdown, no extra text.
Use exactly these keys:
  "language": The programming language name
  "year_created": Year the language was created
  "best_for": What the language is best used for, in one sentence


In [30]:
from langchain_core.output_parsers import JsonOutputParser

struct_prompt = ChatPromptTemplate.from_messages([
    ("system", "You provide technical facts. Follow the output format exactly.\n{format_instructions}"),
    ("human", "Tell me about {language}.")
])

struct_chain = struct_prompt | llm | JsonOutputParser()

result = struct_chain.invoke({
    "language": "Python",
    "format_instructions": format_instructions
})

print(f"Type:    {type(result).__name__}")

if isinstance(result, dict):
    for key, value in result.items():
        print(f"  {key}: {value}")
else:
    print(f"Model returned raw text instead of JSON:")
    print(result)

Type:    str
Model returned raw text instead of JSON:
language


---

## 4. Putting It All Together — The Fundamental Chain Pattern

The three components form the **atom** of LangChain:

```
Prompt Template  →  Chat Model  →  Output Parser
```

In LCEL:
```python
chain = prompt | model | parser
result = chain.invoke({"variable": "value"})
```

Every complex application is built by **combining, nesting, and extending** these atoms.

### Experiment 4A: Complete Chain — All Three Components

In [31]:
# A complete chain demonstrating the fundamental pattern

# 1. PROMPT — the instructions
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert {domain} instructor. "
     "Explain concepts clearly and concisely."),
    ("human", "Explain {concept} in {length}.")
])

# 2. MODEL — the AI brain
model = ChatOllama(model=MODEL, temperature=0.3)

# 3. PARSER — the response formatter
parser = StrOutputParser()

# CHAIN = prompt | model | parser
chain = prompt | model | parser

# Use it
result = chain.invoke({
    "domain": "AI/ML",
    "concept": "the prompt → model → parser pattern",
    "length": "3 sentences"
})

print("Result (clean string):")
display(Markdown(result))

Result (clean string):


The prompt → model → parser pattern is a design pattern used in natural language processing (NLP) that involves three main components: a **prompt** (input text), a **model** (language understanding and generation capabilities), and a **parser** (text analysis and interpretation). The prompt is input into the model, which generates an output based on the input. The parser then analyzes the generated output to extract relevant information or perform specific tasks, such as question answering or sentiment analysis.

### Experiment 4B: Multi-Step Chain — Compose Two Chains

In [32]:
# Step 1: Generate a technical explanation
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Explain in 2-3 sentences."),
    ("human", "Explain: {topic}")
])
tech_chain = tech_prompt | llm | StrOutputParser()

# Step 2: Convert to a fun analogy
analogy_prompt = ChatPromptTemplate.from_messages([
    ("system", "You turn technical explanations into fun cooking analogies. 2 sentences max."),
    ("human", "Make this a cooking analogy: {text}")
])
analogy_chain = analogy_prompt | llm | StrOutputParser()

# Run in sequence
topic = "Output Parsers in LangChain"

print("Step 1 — Technical Explanation:")
print("=" * 50)
tech_explanation = tech_chain.invoke({"topic": topic})
display(Markdown(tech_explanation))

print("\nStep 2 — Cooking Analogy:")
print("=" * 50)
analogy = analogy_chain.invoke({"text": tech_explanation})
display(Markdown(analogy))

Step 1 — Technical Explanation:


**Output Parsers in LangChain**

In the context of natural language processing (NLP) and task-oriented interfaces, an **output parser** is a critical component that enables systems to extract relevant information from user input. In LangChain, an open-source library for building conversational AI models, output parsers play a crucial role in facilitating accurate and efficient understanding of user requests.

**What does an Output Parser do?**

An output parser is responsible for parsing the output of a task model, such as a question answering model or a text classification model. Its primary function is to identify and extract relevant information from the output, which can then be used to inform subsequent actions or responses in the conversation.

**How do Output Parsers work in LangChain?**

In LangChain, an output parser is integrated into the `TaskModel` component, which represents a task-oriented interface. When a user inputs a request, the `TaskModel` processes it and generates an output response. The output parser then takes this output as input and applies various techniques to extract relevant information.

The output parser typically performs the following tasks:

1. **Tokenization**: Breaking down the output into individual tokens or words.
2. **Part-of-speech (POS) tagging**: Identifying the part of speech (e.g., noun, verb, adjective) for each token.
3. **Named entity recognition (NER)**: Identifying named entities such as people, places, and organizations.
4. **Dependency parsing**: Analyzing the grammatical structure of the output to identify relationships between tokens.

**Benefits of Output Parsers in LangChain**

The use of output parsers in LangChain offers several benefits:

1. **Improved accuracy**: By extracting relevant information from the output, output parsers can improve the overall accuracy of the task model's responses.
2. **Increased efficiency**: Output parsers can streamline the processing of output responses, enabling faster and more efficient conversation flows.
3. **Enhanced user experience**: By providing users with more accurate and relevant information, output parsers can enhance the overall conversational experience.

**Examples of Output Parsers in LangChain**

Some examples of output parsers in LangChain include:

1. **NLTK's spaCy parser**: A popular NLP library that provides high-performance, streamlined processing of natural language text.
2. **Stanford CoreNLP's Part-of-Speech Tagger**: A widely used POS tagger that can be integrated into LangChain to analyze the grammatical structure of output responses.

In summary, output parsers in LangChain play a critical role in facilitating accurate and efficient understanding of user input by extracting relevant information from task model outputs. By integrating advanced NLP techniques, such as tokenization, POS tagging, and dependency parsing, output parsers can improve the overall accuracy and efficiency of conversational AI systems.


Step 2 — Cooking Analogy:


**The Recipe for Better Conversations: Output Parsers in LangChain**

Imagine you're a master chef, trying to create the perfect dish based on your customer's order. You need to extract the relevant ingredients (information) from their request, understand what they want (the output), and then use that information to whip up a delicious response.

In natural language processing (NLP) and task-oriented interfaces, an **output parser** is like the sous chef who takes the output of your main recipe (task model) and breaks it down into manageable ingredients. Their primary job is to identify and extract relevant information from the output, which can then be used to inform subsequent actions or responses in the conversation.

**How do Output Parsers work?**

In LangChain, an open-source library for building conversational AI models, output parsers are integrated into the `TaskModel` component, like a special ingredient that enhances the recipe. When a user inputs a request, the `TaskModel` processes it and generates an output response. The output parser then takes this output as input and applies various techniques to extract relevant information.

The output parser typically performs the following tasks:

1. **Tokenization**: Breaking down the output into individual ingredients (tokens or words).
2. **Part-of-speech (POS) tagging**: Identifying the type of each ingredient (e.g., noun, verb, adjective).
3. **Named entity recognition (NER)**: Identifying specific ingredients like people, places, and organizations.
4. **Dependency parsing**: Analyzing the grammatical structure of the output to understand relationships between ingredients.

**Benefits of Output Parsers in LangChain**

The use of output parsers in LangChain offers several benefits:

1. **Improved accuracy**: By extracting relevant information from the output, output parsers can improve the overall accuracy of the task model's responses.
2. **Increased efficiency**: Output parsers can streamline the processing of output responses, enabling faster and more efficient conversation flows.
3. **Enhanced user experience**: By providing users with more accurate and relevant information, output parsers can enhance the overall conversational experience.

**Examples of Output Parsers in LangChain**

Some examples of output parsers in LangChain include:

1. **NLTK's spaCy parser**: A popular NLP library that provides high-performance, streamlined processing of natural language text.
2. **Stanford CoreNLP's Part-of-Speech Tagger**: A widely used POS tagger that can be integrated into LangChain to analyze the grammatical structure of output responses.

In summary, output parsers in LangChain are like skilled sous chefs who help you extract relevant information from your task model's output, enabling better conversations and more accurate responses.

### Experiment 4C: Structured Output Chain — End to End

In [34]:
# Complete example: prompt → model → pydantic parser

class ConceptCard(BaseModel):
    name: str = Field(description="The concept name")
    category: str = Field(description="Category: model, prompt, or parser")
    one_liner: str = Field(description="One sentence explanation")
    when_to_use: str = Field(description="When you would use this, in one sentence")

card_parser = PydanticOutputParser(pydantic_object=ConceptCard)

card_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a LangChain documentation writer. "
     "Respond ONLY with valid JSON. No markdown, no extra text.\n"
     "You MUST use EXACTLY this format, no other keys:\n"
     '{{"name": "...", "category": "model or prompt or parser", '
     '"one_liner": "...", "when_to_use": "..."}}'),
    ("human", "Create a concept card for: {concept}")
])

card_chain = card_prompt | llm | card_parser

concepts = ["ChatPromptTemplate", "StrOutputParser", "ChatOllama"]

for concept in concepts:
    print(f"\n{'=' * 50}")
    try:
        card = card_chain.invoke({
            "concept": concept,
        })
        print(f"  Name:     {card.name}")
        print(f"  Category: {card.category}")
        print(f"  What:     {card.one_liner}")
        print(f"  When:     {card.when_to_use}")
    except Exception as e:
        print(f"  Error: {e}")


  Name:     ChatPromptTemplate
  Category: parser
  What:     A template to structure conversational dialogue with predefined patterns for more coherent and informative responses.
  When:     To create consistent, well-structured conversation flows in chatbots or virtual assistants.

  Name:     StrOutputParser
  Category: parser
  What:     A parser that generates human-readable output from structured data.
  When:     When working with complex data structures or APIs that return JSON or XML output, when a user-friendly output is required.

  Name:     ChatOllama
  Category: language model
  What:     A conversational AI model that leverages the capabilities of LLaMA to engage users in natural-sounding conversations.
  When:     Conversational interfaces, chatbots, and dialogue systems.


---

## 5. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Choose your parser
use_parser = "string"  # Options: "string", "list", "json"

if use_parser == "string":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    my_chain = my_prompt | llm | StrOutputParser()
    result = my_chain.invoke({"question": "What are the 3 core components in LangChain?"})

elif use_parser == "list":
    lp = CommaSeparatedListOutputParser()
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "List items separated by commas only. {fi}"),
        ("human", "List 4 {category}.")
    ])
    my_chain = my_prompt | llm | lp
    result = my_chain.invoke({"category": "output parser types", "fi": lp.get_format_instructions()})

elif use_parser == "json":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "Respond with valid JSON only. No markdown."),
        ("human", 'Describe {topic}. JSON keys: "name", "purpose", "example_use".')
    ])
    my_chain = my_prompt | llm | JsonOutputParser()
    result = my_chain.invoke({"topic": "ChatPromptTemplate"})

print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Models** | Unified interface across all providers — write once, swap freely |
| **Chat Models** | Message-based (SystemMessage, HumanMessage, AIMessage) — the modern standard |
| **Runnable Protocol** | Every component supports `invoke()`, `stream()`, `batch()` |
| **Temperature** | 0.0 = deterministic, 0.5 = balanced, 1.0 = creative |
| **Swappability** | Change one import line to switch providers — chain logic stays identical |
| **PromptTemplate** | Simple string templates with `{variable}` placeholders |
| **ChatPromptTemplate** | Role-based message templates — use this 90%+ of the time |
| **MessagesPlaceholder** | Inject dynamic conversation history into prompts |
| **FewShotChatMessagePromptTemplate** | Teach by example when zero-shot fails |
| **StrOutputParser** | AIMessage → clean string |
| **CommaSeparatedListOutputParser** | Text → Python list |
| **JsonOutputParser** | Text → Python dict (handles code fences) |
| **PydanticOutputParser** | Text → validated Pydantic model (production-grade) |
| **The Pattern** | `prompt \| model \| parser` — the atom of every LangChain app |